In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr
import re
import warnings
warnings.filterwarnings('ignore') # Чтобы не было предупреждений от scipy

# Шаг 1: Создание набора данных (смесь коротких, средних и длинных текстов)
qa_pairs = [
    ("Каков ваш график работы?", "Мы работаем с понедельника по пятницу с 9:00 до 18:00."),
    ("Как получить пропуск в университет?", "Для получения пропуска нужно обратиться в отдел безопасности с паспортом и заявлением. Процесс занимает 3 рабочих дня."),
    ("Где находится библиотека?", "В главном корпусе на втором этаже."), # КОРОТКИЙ
    ("Как записаться на курс?", "Запись на курс осуществляется через личный кабинет студента на сайте университета. Вам нужно зайти в раздел 'Дисциплины', выбрать нужный курс и нажать кнопку 'Записаться'. Если возникнут проблемы, обратитесь в учебный отдел."), # ДЛИННЫЙ
    ("Какие документы нужны для поступления?", "Паспорт, аттестат, фото и медсправка."), # КОРОТКИЙ
    ("Сколько стоит обучение?", "Стоимость обучения зависит от выбранной программы и составляет от 150 до 300 тысяч рублей в год."),
    ("Есть ли общежитие?", "Да, университет предоставляет общежитие для иногородних студентов. Стоимость проживания составляет около 5 тысяч рублей в месяц. Для заселения необходимо подать заявление в деканат до 1 августа."), # ДЛИННЫЙ
    ("Как получить стипендию?", "Стипендия назначается студентам с высокой успеваемостью по результатам сессии."),
    ("Какие факультеты есть в университете?", "ИТ, экономики, инженерии и гуманитарных наук."), # КОРОТКИЙ
    ("Как связаться с деканатом?", "С деканатом можно связаться по телефону или электронной почте, указанной на сайте факультета."),
    ("Когда начинаются каникулы?", "Зимние — в конце декабря, летние — в июне."), # КОРОТКИЙ
    ("Как оформить академический отпуск?", "Академический отпуск оформляется по заявлению студента с указанием причины. Вам потребуется собрать справку из медицинского учреждения или повестку из военкомата. После этого документы подаются в деканат, где их рассматривают в течение 10 дней."), # ДЛИННЫЙ
    ("Где находится столовая?", "В первом корпусе."), # ОЧЕНЬ КОРОТКИЙ
    ("Какие спортивные секции есть?", "Футбол, баскетбол, плавание и единоборства."), # КОРОТКИЙ
    ("Как получить диплом с отличием?", "Для получения диплома с отличием нужно иметь средний балл не ниже 4.75 и сдать все экзамены на отлично.")
]

df_qa = pd.DataFrame(qa_pairs, columns=['Вопрос', 'Ответ'])

# Красивый вывод базы данных
print("="*80)
print("БАЗА ЗНАНИЙ (ВОПРОС-ОТВЕТ):")
print("="*80)
for i, row in df_qa.iterrows():
    print(f"{i+1}. ВОПРОС: {row['Вопрос']}")
    print(f"   ОТВЕТ:  {row['Ответ']}\n")

БАЗА ЗНАНИЙ (ВОПРОС-ОТВЕТ):
1. ВОПРОС: Каков ваш график работы?
   ОТВЕТ:  Мы работаем с понедельника по пятницу с 9:00 до 18:00.

2. ВОПРОС: Как получить пропуск в университет?
   ОТВЕТ:  Для получения пропуска нужно обратиться в отдел безопасности с паспортом и заявлением. Процесс занимает 3 рабочих дня.

3. ВОПРОС: Где находится библиотека?
   ОТВЕТ:  В главном корпусе на втором этаже.

4. ВОПРОС: Как записаться на курс?
   ОТВЕТ:  Запись на курс осуществляется через личный кабинет студента на сайте университета. Вам нужно зайти в раздел 'Дисциплины', выбрать нужный курс и нажать кнопку 'Записаться'. Если возникнут проблемы, обратитесь в учебный отдел.

5. ВОПРОС: Какие документы нужны для поступления?
   ОТВЕТ:  Паспорт, аттестат, фото и медсправка.

6. ВОПРОС: Сколько стоит обучение?
   ОТВЕТ:  Стоимость обучения зависит от выбранной программы и составляет от 150 до 300 тысяч рублей в год.

7. ВОПРОС: Есть ли общежитие?
   ОТВЕТ:  Да, университет предоставляет общежитие для иногор

In [3]:
import pymorphy3

# Инициализация лемматизатора для русского языка
morph = pymorphy3.MorphAnalyzer()

def preprocess_text(text):
    """Правильная предобработка текста с лемматизацией"""
    # Приведение к нижнему регистру
    text = text.lower()
    # Удаление знаков препинания
    text = re.sub(r'[^\w\s]', '', text)
    # Расширенный список стоп-слов
    stop_words = {'и', 'в', 'не', 'на', 'я', 'быть', 'с', 'у', 'он', 'она', 'они', 'мы', 'вы', 'это', 'как', 'что', 'где', 'когда', 'кто', 'по', 'для', 'из', 'от', 'до', 'за', 'через', 'после', 'перед', 'о', 'об', 'а', 'но', 'или', 'же', 'ли', 'то', 'так', 'вот', 'все', 'весь', 'вся', 'всё', 'каков', 'ваш'}
    
    words = text.split()
    # Лемматизация: приводим каждое слово к начальной форме (например, "работаете" -> "работать", "работы" -> "работа")
    words = [morph.parse(word)[0].normal_form for word in words if word not in stop_words]
    
    return ' '.join(words)

# Применяем обработку
df_qa['Вопрос_обработанный'] = df_qa['Вопрос'].apply(preprocess_text)
df_qa['Ответ_обработанный'] = df_qa['Ответ'].apply(preprocess_text)

print("После правильной предобработки (лемматизации):")
print(df_qa[['Вопрос_обработанный']].head())

После правильной предобработки (лемматизации):
                 Вопрос_обработанный
0                      график работа
1       получить пропуск университет
2              находиться библиотека
3                    записаться курс
4  какой документ нужный поступление


In [4]:

# Шаг 3: Bag of Words (Мешок слов)
vectorizer = CountVectorizer()
X_questions = vectorizer.fit_transform(df_qa['Вопрос_обработанный'])
X_answers = vectorizer.transform(df_qa['Ответ_обработанный'])

print("Матрица Bag of Words для вопросов:")
print("Размерность:", X_questions.shape)
print("Словарь:", vectorizer.get_feature_names_out()[:10], "...")
print("\n")



Матрица Bag of Words для вопросов:
Размерность: (15, 32)
Словарь: ['академический' 'библиотека' 'график' 'деканат' 'диплом' 'документ'
 'есть' 'записаться' 'какой' 'каникулы'] ...




In [5]:
# Шаг 4: Функции для вычисления сходства
def jaccard_similarity(vec1, vec2):
    """Сходство Жаккара"""
    intersection = np.minimum(vec1, vec2).sum()
    union = np.maximum(vec1, vec2).sum()
    if union == 0:
        return 0
    return intersection / union

def cosine_distance(vec1, vec2):
    """Косинусное расстояние (1 - косинусное сходство)"""
    similarity = cosine_similarity(vec1.reshape(1, -1), vec2.reshape(1, -1))[0][0]
    return 1 - similarity

def pearson_correlation(vec1, vec2):
    """Корреляция Пирсона"""
    if np.std(vec1) == 0 or np.std(vec2) == 0:
        return 0
    correlation, _ = pearsonr(vec1, vec2)
    return correlation

# Функция поиска наиболее близкого вопроса
def find_best_match(query, df_qa, vectorizer):
    """Поиск наиболее близкого вопроса в базе знаний"""
    # Предобработка запроса
    query_processed = preprocess_text(query)
    # Векторизация запроса
    query_vector = vectorizer.transform([query_processed]).toarray()[0]
    
    results = []
    for idx, row in df_qa.iterrows():
        question_vector = X_questions[idx].toarray()[0]
        
        # Вычисление метрик сходства
        jaccard = jaccard_similarity(query_vector, question_vector)
        cosine_dist = cosine_distance(query_vector, question_vector)
        pearson_corr = pearson_correlation(query_vector, question_vector)
        
        results.append({
            'Индекс': idx,
            'Вопрос': row['Вопрос'],
            'Ответ': row['Ответ'],
            'Жаккар': jaccard,
            'Косинус': cosine_dist,
            'Пирсон': pearson_corr
        })
    
    return pd.DataFrame(results)



In [6]:
# Шаг 5: Подготовка тестовых вопросов
test_questions_with_answers = [
    "Когда вы работаете?",
    "Как попасть в университет?",
    "Где можно найти книги?",
    "Как зарегистрироваться на занятия?",
    "Что нужно для поступления?",
    "Сколько нужно платить за учебу?",
    "Предоставляется ли жилье?",
    "Как получить материальную помощь?",
    "Какие направления обучения есть?",
    "Как найти деканат?"
]

test_questions_without_answers = [
    "Какая погода сегодня?",
    "Сколько времени?",
    "Где ближайший магазин?",
    "Как приготовить борщ?",
    "Какой сегодня день недели?"
]


In [7]:
results_with_answers = []
for i, question in enumerate(test_questions_with_answers):
    matches = find_best_match(question, df_qa, vectorizer)
    
    # Находим лучший ответ по каждой метрике
    best_jaccard = matches.loc[matches['Жаккар'].idxmax()]
    best_cosine = matches.loc[matches['Косинус'].idxmin()] # Меньше расстояние = лучше
    best_pearson = matches.loc[matches['Пирсон'].idxmax()]
    
    # Берем правильный ответ из базы (предполагая, что порядок вопросов совпадает)
    correct_answer = df_qa.iloc[i]['Ответ'] if i < len(df_qa) else "Нет в базе"
    
    results_with_answers.append({
        'Вопрос': question,
        'Правильный ответ': correct_answer,
        'Ответ_Жаккар': best_jaccard['Ответ'],
        'Ответ_Косинус': best_cosine['Ответ'],
        'Ответ_Пирсон': best_pearson['Ответ'],
        'Метрика_Жаккар': best_jaccard['Жаккар'],
        'Метрика_Косинус': best_cosine['Косинус'],
        'Метрика_Пирсон': best_pearson['Пирсон']
    })

df_results_with = pd.DataFrame(results_with_answers)

# 2. Тестирование на вопросах БЕЗ ответов
results_without_answers = []
for question in test_questions_without_answers:
    matches = find_best_match(question, df_qa, vectorizer)
    
    best_jaccard = matches.loc[matches['Жаккар'].idxmax()]
    best_cosine = matches.loc[matches['Косинус'].idxmin()]
    best_pearson = matches.loc[matches['Пирсон'].idxmax()]
    
    results_without_answers.append({
        'Вопрос': question,
        'Правильный ответ': "Нет в базе",
        'Ответ_Жаккар': best_jaccard['Ответ'],
        'Ответ_Косинус': best_cosine['Ответ'],
        'Ответ_Пирсон': best_pearson['Ответ'],
        'Метрика_Жаккар': best_jaccard['Жаккар'],
        'Метрика_Косинус': best_cosine['Косинус'],
        'Метрика_Пирсон': best_pearson['Пирсон']
    })

df_results_without = pd.DataFrame(results_without_answers)

In [8]:
def print_pretty_table(df, title):
    print(f"\n{'='*100}\n{title}\n{'='*100}")
    for i, row in df.iterrows():
        print(f"Вопрос: {row['Вопрос']}")
        print(f"  Правильный ответ: {row['Правильный ответ']}")
        print(f"  [Жаккар]  Ответ: {row['Ответ_Жаккар']} \n            (Метрика: {row['Метрика_Жаккар']:.4f})")
        print(f"  [Косинус] Ответ: {row['Ответ_Косинус']} \n            (Расстояние: {row['Метрика_Косинус']:.4f})")
        print(f"  [Пирсон]  Ответ: {row['Ответ_Пирсон']} \n            (Корреляция: {row['Метрика_Пирсон']:.4f})")
        print("-" * 100)

# Выводим результаты
print_pretty_table(df_results_with, "ТЕСТИРОВАНИЕ НА ВОПРОСАХ С ОТВЕТАМИ В БАЗЕ")
print_pretty_table(df_results_without, "ТЕСТИРОВАНИЕ НА ВОПРОСАХ БЕЗ ОТВЕТОВ В БАЗЕ")


ТЕСТИРОВАНИЕ НА ВОПРОСАХ С ОТВЕТАМИ В БАЗЕ
Вопрос: Когда вы работаете?
  Правильный ответ: Мы работаем с понедельника по пятницу с 9:00 до 18:00.
  [Жаккар]  Ответ: Мы работаем с понедельника по пятницу с 9:00 до 18:00. 
            (Метрика: 0.0000)
  [Косинус] Ответ: Мы работаем с понедельника по пятницу с 9:00 до 18:00. 
            (Расстояние: 1.0000)
  [Пирсон]  Ответ: Мы работаем с понедельника по пятницу с 9:00 до 18:00. 
            (Корреляция: 0.0000)
----------------------------------------------------------------------------------------------------
Вопрос: Как попасть в университет?
  Правильный ответ: Для получения пропуска нужно обратиться в отдел безопасности с паспортом и заявлением. Процесс занимает 3 рабочих дня.
  [Жаккар]  Ответ: Для получения пропуска нужно обратиться в отдел безопасности с паспортом и заявлением. Процесс занимает 3 рабочих дня. 
            (Метрика: 0.3333)
  [Косинус] Ответ: Для получения пропуска нужно обратиться в отдел безопасности с паспор

In [9]:

# Шаг 7: Анализ и выводы
print("\n" + "=" * 80)
print("АНАЛИЗ РЕЗУЛЬТАТОВ")
print("=" * 80)

# Средние значения метрик для вопросов с ответами
avg_jaccard_with = df_results_with['Метрика_Жаккар'].mean()
avg_cosine_with = df_results_with['Метрика_Косинус'].mean()
avg_pearson_with = df_results_with['Метрика_Пирсон'].mean()

# Средние значения метрик для вопросов без ответов
avg_jaccard_without = df_results_without['Метрика_Жаккар'].mean()
avg_cosine_without = df_results_without['Метрика_Косинус'].mean()
avg_pearson_without = df_results_without['Метрика_Пирсон'].mean()

print(f"\nСредние метрики для вопросов С ответами:")
print(f"  Жаккар: {avg_jaccard_with:.4f}")
print(f"  Косинус (расстояние): {avg_cosine_with:.4f}")
print(f"  Пирсон: {avg_pearson_with:.4f}")

print(f"\nСредние метрики для вопросов БЕЗ ответов:")
print(f"  Жаккар: {avg_jaccard_without:.4f}")
print(f"  Косинус (расстояние): {avg_cosine_without:.4f}")
print(f"  Пирсон: {avg_pearson_without:.4f}")




АНАЛИЗ РЕЗУЛЬТАТОВ

Средние метрики для вопросов С ответами:
  Жаккар: 0.2317
  Косинус (расстояние): 0.6354
  Пирсон: 0.3510

Средние метрики для вопросов БЕЗ ответов:
  Жаккар: 0.1667
  Косинус (расстояние): 0.6845
  Пирсон: 0.3018
